# Knowledge Engineering
## Autonomous Trailer: A Knowledge-Based Agent for Kenya's Highway Network

**Course:** ICS 3202 [Aug–Nov 2026], Practical Exercise 4

**Group members:** Anthony Mutua, Emmanuel Bosire, Gloria Muluka

### Evidence of Group Work

![Team photo](team_photo.png)

*Online group session on WhatsApp. All three members who worked on the assignment are visible. The same image is also uploaded to this repository as `team_photo.png`.*

---

**Step 1: Identify the Task**

The task is to help a self-driving trailer decide **which route to take** between two towns on Kenya's highway network and **how to drive each stretch** (speed, signals, road closures) based on what the agent knows.

**Goal:** Deliver cargo from the start town to the destination along the shortest *passable* route, while driving safely for the conditions on each road segment.

**Step 2: Assemble Relevant Knowledge**

The agent needs to know:
*   Which towns exist and which region each belongs to (Nairobi, Central Highlands, Rift Valley, Lake Victoria Basin)
*   Which towns are joined by a highway, the distance (km) and the highway name
*   Which roads are blocked, which have hazards (e.g. escarpments), and what the weather is on each road
*   The correct action for each traffic light colour

**Sources of knowledge:** The directed graph, search tree and distance measures from Practical Exercise 3 (`design.md`), plus general driving rules written by the group. The distances are approximate driving distances between town centres.

**APIs / web scraping:** No APIs or web scraping tools were used. All knowledge was encoded manually from Practical Exercise 3.

**Step 3: Define the Vocabulary**

*   **Predicates** (represented in Python as dictionaries, sets and rules):
    *   `InRegion(town, region)`
    *   `Road(a, b, km, highway)`
    *   `Blocked(a, b)`
    *   `Hazard(a, b)`
    *   `Weather(a, b)`
    *   `Light(colour)`
    *   `Passable(a, b)` *(derived by a rule)*
    *   `SpeedLimit(a, b)` *(derived by a rule)*
*   **Constants:**
    *   Towns: Nairobi, Thika, Murang'a, Nyeri, Nanyuki, Naivasha, Nakuru, Nyahururu, Molo, Kericho, Eldoret, Kisumu, Kakamega, Kisii
    *   Regions: Nairobi, Central Highlands, Rift Valley, Lake Victoria Basin
    *   Weather: Clear, Rain, Fog
    *   Lights: Red, Yellow, Green

**Rules (general knowledge):**
1. `Passable(a, b)` ⇐ `Road(a, b)` ∧ ¬`Blocked(a, b)`
2. The trailer's best route is the passable path with the lowest total distance
3. `SpeedLimit(a, b)` depends on `Weather(a, b)` and whether `Hazard(a, b)` exists
4. The action to take at a traffic light depends on its colour

**Step 4: Encode General Knowledge about the Domain**

In [14]:
import heapq
import copy

class KnowledgeBase:
    """Stores facts (InRegion, Road, Blocked, Hazard, Weather, Light) and applies rules to them."""

    # Rule 3: speed limit (km/h) from (weather, hazard present?). Values are assumed by the group.
    SPEED_RULES = {
        ("Clear", False): 80, ("Clear", True): 60,
        ("Rain",  False): 60, ("Rain",  True): 40,
        ("Fog",   False): 40, ("Fog",   True): 30,
    }

    def __init__(self):
        self.in_region = {}    # InRegion(town, region)
        self.roads = {}        # Road(a, b) -> (km, highway); stored in both directions
        self.blocked = set()   # Blocked(a, b)
        self.hazard = {}       # Hazard(a, b) -> description
        self.weather = {}      # Weather(a, b) -> condition
        self.light_rules = {}  # Light(colour) -> action

    # ---------- TELL: add facts ----------
    def tell_town(self, town, region):
        self.in_region[town] = region

    def tell_road(self, a, b, km, highway):
        self.roads[(a, b)] = (km, highway)
        self.roads[(b, a)] = (km, highway)

    def tell_blocked(self, a, b, blocked=True):
        for pair in ((a, b), (b, a)):
            if blocked:
                self.blocked.add(pair)
            else:
                self.blocked.discard(pair)

    def tell_hazard(self, a, b, description):
        self.hazard[(a, b)] = self.hazard[(b, a)] = description

    def tell_weather(self, a, b, condition):
        self.weather[(a, b)] = self.weather[(b, a)] = condition

    def tell_light(self, colour, action):
        self.light_rules[colour.capitalize()] = action

    # ---------- RULES: derive new knowledge ----------
    def passable(self, a, b):
        """Rule 1: Passable(a, b) <= Road(a, b) AND NOT Blocked(a, b)"""
        return (a, b) in self.roads and (a, b) not in self.blocked

    def neighbours(self, town):
        return [(b, self.roads[(a, b)][0]) for (a, b) in self.roads
                if a == town and self.passable(a, b)]

    def speed_limit(self, a, b):
        """Rule 3: SpeedLimit(a, b) from Weather(a, b) and Hazard(a, b)"""
        condition = self.weather.get((a, b), "Clear")
        return self.SPEED_RULES.get((condition, (a, b) in self.hazard), 40)

    def light_advice(self, colour):
        """Rule 4: what to do at a traffic light"""
        return self.light_rules.get(colour.capitalize(),
                                    "Unknown signal: slow down and proceed with caution or stop.")

kb = KnowledgeBase()

# InRegion(town, region): from the Practical Exercise 3 region table
regions = {
    "Nairobi": ["Nairobi"],
    "Central Highlands": ["Thika", "Murang'a", "Nyeri", "Nanyuki"],
    "Rift Valley": ["Naivasha", "Nakuru", "Nyahururu", "Molo", "Kericho", "Eldoret"],
    "Lake Victoria Basin": ["Kisumu", "Kakamega", "Kisii"],
}
for region, towns in regions.items():
    for town in towns:
        kb.tell_town(town, region)

# Road(a, b, km, highway): from the Practical Exercise 3 distance table
road_facts = [
    ("Nairobi", "Thika", 42, "A2"),          ("Thika", "Murang'a", 40, "A2"),
    ("Murang'a", "Nyeri", 55, "A2"),         ("Nyeri", "Nanyuki", 62, "A2"),
    ("Nyeri", "Nyahururu", 85, "B5 / C-roads"), ("Nairobi", "Naivasha", 92, "A104"),
    ("Naivasha", "Nakuru", 64, "A104"),      ("Nakuru", "Nyahururu", 90, "B5"),
    ("Nakuru", "Molo", 50, "A104"),          ("Molo", "Kericho", 65, "A104 / A1"),
    ("Kericho", "Kisumu", 80, "A1"),         ("Kericho", "Kisii", 100, "B-roads"),
    ("Kisii", "Kisumu", 117, "A1 / B-roads"), ("Nakuru", "Eldoret", 155, "A104"),
    ("Eldoret", "Kakamega", 85, "A1 / B-roads"), ("Kakamega", "Kisumu", 52, "A1"),
]
for a, b, km, highway in road_facts:
    kb.tell_road(a, b, km, highway)

# Hazard(a, b): steep escarpment sections that the trailer must treat carefully
kb.tell_hazard("Nairobi", "Naivasha", "Mai Mahiu escarpment: steep descent")
kb.tell_hazard("Molo", "Kericho", "Mau Summit: steep climb, fog-prone")

# Light(colour): from the traffic light knowledge base
kb.tell_light("Red", "Stop: it's unsafe to move.")
kb.tell_light("Yellow", "Slow down: prepare to stop.")
kb.tell_light("Green", "Go: it's safe to proceed.")

print(f"Towns: {len(kb.in_region)} | Road segments: {len(kb.roads) // 2} | Hazards: {len(kb.hazard) // 2} | Light rules: {len(kb.light_rules)}")

Towns: 14 | Road segments: 16 | Hazards: 2 | Light rules: 3


In [15]:
def plan_route(kb, start, goal):
    """Rule 2: Uniform-Cost Search over passable roads (path cost = distance in km).
    Returns (path, total_km, expansion_order), or (None, None, expansion_order) if no route exists."""
    for town in (start, goal):
        if town not in kb.in_region:
            raise ValueError(f"Unknown town: '{town}'. The knowledge base has no facts about it.")
    frontier = [(0, start, [start])]
    best = {}
    expanded = []
    while frontier:
        g, town, path = heapq.heappop(frontier)
        if town in best and best[town] <= g:
            continue
        best[town] = g
        expanded.append((town, g))
        if town == goal:                      # goal test on expansion guarantees optimality
            return path, g, expanded
        for nxt, km in kb.neighbours(town):
            if nxt not in best:
                heapq.heappush(frontier, (g + km, nxt, path + [nxt]))
    return None, None, expanded

def explain_route(kb, path):
    """Describe how to drive each leg, using the speed and hazard rules."""
    total = 0
    for a, b in zip(path, path[1:]):
        km, highway = kb.roads[(a, b)]
        total += km
        note = f" | Hazard: {kb.hazard[(a, b)]}" if (a, b) in kb.hazard else ""
        weather = kb.weather.get((a, b), "Clear")
        print(f"{a:10} -> {b:10} {km:4} km  {highway:13} {weather:5} Max {kb.speed_limit(a, b)} km/h{note}")
    print(f"{'TOTAL':>25} {total:4} km")

**Step 5: Add New Facts to the Knowledge Base** 

In [16]:
def show_kb():
    print("\n--- Current Knowledge Base ---")
    print(f"Towns ({len(kb.in_region)}):")
    for region in dict.fromkeys(kb.in_region.values()):
        towns = [t for t, r in kb.in_region.items() if r == region]
        print(f"  {region:20} {', '.join(towns)}")
    print(f"Roads ({len(kb.roads) // 2}):")
    for (a, b), (km, hw) in kb.roads.items():
        if a < b:
            flag = "  [BLOCKED]" if (a, b) in kb.blocked else ""
            print(f"  {a:10} <-> {b:10} {km:4} km  {hw}{flag}")
    print("Traffic lights:")
    for light, action in kb.light_rules.items():
        print(f"  {light:10} -> {action}")
    print("-------------------------------")

def add_new_fact(town_a, region_a, town_b, region_b, km, highway):
    """Add two towns and the road between them directly to the KB (without user input)."""
    kb.tell_town(town_a, region_a)
    kb.tell_town(town_b, region_b)
    kb.tell_road(town_a, town_b, km, highway)
    print(f"Added fact: Road({town_a}, {town_b}, {km} km, {highway})")

# Example of adding facts: a southern route through Narok and Bomet to Kericho
add_new_fact("Nairobi", "Nairobi", "Narok", "Rift Valley", 145, "A104 / B3")
add_new_fact("Narok", "Rift Valley", "Bomet", "Rift Valley", 95, "B-roads")
add_new_fact("Bomet", "Rift Valley", "Kericho", "Rift Valley", 55, "B-roads")

# New light rules
kb.tell_light("Flashing", "Stop and check the intersection.")
kb.tell_light("Blinking", "Proceed slowly with caution.")

def add_fact_interactive():
    """Add a new road to the KB from keyboard input (run this cell function in Colab to try it)."""
    a = input("First town: ").strip().capitalize()
    b = input("Second town: ").strip().capitalize()
    km = float(input(f"Distance between {a} and {b} (km): "))
    highway = input("Highway name: ").strip()
    for town in (a, b):
        if town not in kb.in_region:
            kb.tell_town(town, input(f"Region of {town}: ").strip())
    kb.tell_road(a, b, km, highway)
    print(f"New fact added: Road({a}, {b}, {km} km, {highway})")

# add_fact_interactive()   # <- uncomment to add a fact by typing it in

# View updated KB
show_kb()

Added fact: Road(Nairobi, Narok, 145 km, A104 / B3)
Added fact: Road(Narok, Bomet, 95 km, B-roads)
Added fact: Road(Bomet, Kericho, 55 km, B-roads)

--- Current Knowledge Base ---
Towns (16):
  Nairobi              Nairobi
  Central Highlands    Thika, Murang'a, Nyeri, Nanyuki
  Rift Valley          Naivasha, Nakuru, Nyahururu, Molo, Kericho, Eldoret, Narok, Bomet
  Lake Victoria Basin  Kisumu, Kakamega, Kisii
Roads (19):
  Nairobi    <-> Thika        42 km  A2
  Murang'a   <-> Thika        40 km  A2
  Murang'a   <-> Nyeri        55 km  A2
  Nanyuki    <-> Nyeri        62 km  A2
  Nyahururu  <-> Nyeri        85 km  B5 / C-roads
  Nairobi    <-> Naivasha     92 km  A104
  Naivasha   <-> Nakuru       64 km  A104
  Nakuru     <-> Nyahururu    90 km  B5
  Molo       <-> Nakuru       50 km  A104
  Kericho    <-> Molo         65 km  A104 / A1
  Kericho    <-> Kisumu       80 km  A1
  Kericho    <-> Kisii       100 km  B-roads
  Kisii      <-> Kisumu      117 km  A1 / B-roads
  Eldoret    <->

**Steps 6 & 7: Encode Specific Problem Instance (Facts) & Pose Queries and Get Answers** 

**Problem instance:** The trailer is in **Nairobi** and must reach **Kisumu**. The sensors report rain on the Nairobi to Naivasha road and the next traffic light is green.

In [17]:
# Step 6: current observations (facts about this specific trip)
start, goal = "Nairobi", "Kisumu"
kb.tell_weather("Nairobi", "Naivasha", "Rain")
current_light = "Green"

# Step 7: the agent queries the knowledge base
path, km, order = plan_route(kb, start, goal)

print("=== Autonomous Trailer Decision System ===")
print(f"Trip: {start} -> {goal}")
print("Optimal route:", " -> ".join(path))
print(f"Total distance: {km} km\n")
explain_route(kb, path)
print(f"\nObserved light: {current_light} | Agent decision: {kb.light_advice(current_light)}")

=== Autonomous Trailer Decision System ===
Trip: Nairobi -> Kisumu
Optimal route: Nairobi -> Naivasha -> Nakuru -> Molo -> Kericho -> Kisumu
Total distance: 351 km

Nairobi    -> Naivasha     92 km  A104          Rain  Max 40 km/h | Hazard: Mai Mahiu escarpment: steep descent
Naivasha   -> Nakuru       64 km  A104          Clear Max 80 km/h
Nakuru     -> Molo         50 km  A104          Clear Max 80 km/h
Molo       -> Kericho      65 km  A104 / A1     Clear Max 60 km/h | Hazard: Mau Summit: steep climb, fog-prone
Kericho    -> Kisumu       80 km  A1            Clear Max 80 km/h
                    TOTAL  351 km

Observed light: Green | Agent decision: Go: it's safe to proceed.


**Step 8: Querying the knowledge base**

In [18]:
# Query 1: another destination
path, km, _ = plan_route(kb, "Nairobi", "Eldoret")
print("Nairobi -> Eldoret:", " -> ".join(path), f"({km} km)")

# Query 2: is a given road passable?
print("\nIs Molo -> Kericho passable?", kb.passable("Molo", "Kericho"))
print("Is Nairobi -> Kisumu a direct road?", kb.passable("Nairobi", "Kisumu"))

# Query 3: how did the search explore the map?
path, km, order = plan_route(kb, "Nairobi", "Kisumu")
print("\nUniform-cost search expansion order (town, g in km):")
print(order)

# Query 4: what to do at different lights?
for light in ["Red", "Flashing", "Teal"]:
    print(f"Light '{light}':", kb.light_advice(light))

# Query 5: the knowledge changes, so the answer changes. A landslide blocks Molo - Kericho.
print("\n--- Landslide reported: Molo <-> Kericho is now BLOCKED ---")
kb.tell_blocked("Molo", "Kericho")
path, km, _ = plan_route(kb, "Nairobi", "Kisumu")
print("New route:", " -> ".join(path), f"({km} km)")

# The road is cleared, so the original route is restored
kb.tell_blocked("Molo", "Kericho", blocked=False)
path, km, _ = plan_route(kb, "Nairobi", "Kisumu")
print("Road cleared. Route:", " -> ".join(path), f"({km} km)")

Nairobi -> Eldoret: Nairobi -> Naivasha -> Nakuru -> Eldoret (311 km)

Is Molo -> Kericho passable? True
Is Nairobi -> Kisumu a direct road? False

Uniform-cost search expansion order (town, g in km):
[('Nairobi', 0), ('Thika', 42), ("Murang'a", 82), ('Naivasha', 92), ('Nyeri', 137), ('Narok', 145), ('Nakuru', 156), ('Nanyuki', 199), ('Molo', 206), ('Nyahururu', 222), ('Bomet', 240), ('Kericho', 271), ('Eldoret', 311), ('Kisumu', 351)]
Light 'Red': Stop: it's unsafe to move.
Light 'Flashing': Stop and check the intersection.
Light 'Teal': Unknown signal: slow down and proceed with caution or stop.

--- Landslide reported: Molo <-> Kericho is now BLOCKED ---
New route: Nairobi -> Narok -> Bomet -> Kericho -> Kisumu (375 km)
Road cleared. Route: Nairobi -> Naivasha -> Nakuru -> Molo -> Kericho -> Kisumu (351 km)


**Step 9: Debugging the Knowledge Base**

In [19]:
def audit(kb):
    """Check the KB for faulty or inconsistent facts."""
    problems = []
    for (a, b), (km, hw) in kb.roads.items():
        if km <= 0:
            problems.append(f"Non-positive distance on {a} - {b}: {km}")
        if kb.roads.get((b, a)) != (km, hw):
            problems.append(f"Road {a} - {b} is not the same in both directions")
        for town in (a, b):
            if town not in kb.in_region:
                problems.append(f"Town '{town}' has no region (road {a} - {b})")
    connected = {t for pair in kb.roads for t in pair}
    for town in kb.in_region:
        if town not in connected:
            problems.append(f"Town '{town}' has no roads (isolated)")
    return sorted(set(problems))

print("--- Auditing the real KB ---")
print(audit(kb) or "No problems found.")

# Test the audit with deliberately faulty facts on a COPY of the KB
faulty = copy.deepcopy(kb)
faulty.tell_road("Eldoret", "Kitale", -70, "A1")   # negative distance, Kitale has no region
faulty.tell_town("Busia", "Lake Victoria Basin")   # town with no roads
print("\n--- Auditing a faulty copy ---")
for p in audit(faulty):
    print(" -", p)

# Fix the faulty facts
faulty.tell_road("Eldoret", "Kitale", 70, "A1")
faulty.tell_town("Kitale", "Rift Valley")
faulty.tell_road("Kisumu", "Busia", 100, "A1")
print("\n--- Auditing after the fix ---")
print(audit(faulty) or "No problems found.")

# Test planning and light queries with tricky inputs
print("\n--- Debugging queries ---")
tests = [("Nairobi", "Nairobi"), ("Nairobi", "Mombasa"), ("Kakamega", "Nanyuki")]
for s, g in tests:
    try:
        path, km, _ = plan_route(kb, s, g)
        print(f"{s} -> {g}:", " -> ".join(path), f"({km} km)")
    except ValueError as error:
        print(f"{s} -> {g}: ERROR: {error}")

# Unreachable destination: block every road into Kisumu on a copy
cut_off = copy.deepcopy(kb)
for neighbour, _ in kb.neighbours("Kisumu"):
    cut_off.tell_blocked("Kisumu", neighbour)
path, km, _ = plan_route(cut_off, "Nairobi", "Kisumu")
print("Nairobi -> Kisumu with every road into Kisumu blocked:", path or "No route found. Wait or ask the operator.")

for light in ["Red", "Yellow", "Green", "Blue", "None", "Purple"]:
    print(f"Light: {light:7} -> Action: {kb.light_advice(light)}")

--- Auditing the real KB ---
No problems found.

--- Auditing a faulty copy ---
 - Non-positive distance on Eldoret - Kitale: -70
 - Non-positive distance on Kitale - Eldoret: -70
 - Town 'Busia' has no roads (isolated)
 - Town 'Kitale' has no region (road Eldoret - Kitale)
 - Town 'Kitale' has no region (road Kitale - Eldoret)

--- Auditing after the fix ---
No problems found.

--- Debugging queries ---
Nairobi -> Nairobi: Nairobi (0 km)
Nairobi -> Mombasa: ERROR: Unknown town: 'Mombasa'. The knowledge base has no facts about it.
Kakamega -> Nanyuki: Kakamega -> Eldoret -> Nakuru -> Nyahururu -> Nyeri -> Nanyuki (477 km)
Nairobi -> Kisumu with every road into Kisumu blocked: No route found. Wait or ask the operator.
Light: Red     -> Action: Stop: it's unsafe to move.
Light: Yellow  -> Action: Slow down: prepare to stop.
Light: Green   -> Action: Go: it's safe to proceed.
Light: Blue    -> Action: Unknown signal: slow down and proceed with caution or stop.
Light: None    -> Action: Un

The debugging run showed that invalid or missing light signals (Blue, None, Purple) fall through to the general "unknown signal" rule. A rule can be added for any of them, for example `kb.tell_light("None", "No signal detected: proceed carefully and keep scanning.")`. The audit also caught a negative distance, a road to a town without a region, and an isolated town, all of which would have produced wrong routes.

In [20]:
# Add specific rules for the signals found during debugging
kb.tell_light("None", "No signal detected: proceed carefully and keep scanning.")
kb.tell_light("Blue", "Not a valid traffic signal: treat as a sensor error and slow down.")
for light in ["None", "Blue", "Purple"]:
    print(f"Light: {light:7} -> Action: {kb.light_advice(light)}")

Light: None    -> Action: No signal detected: proceed carefully and keep scanning.
Light: Blue    -> Action: Not a valid traffic signal: treat as a sensor error and slow down.
Light: Purple  -> Action: Unknown signal: slow down and proceed with caution or stop.


**Step 10: Add final reflection here (challenges or recommended improvements)** 

**Challenges**
*   **Approximate distances.** The route the agent picks is only as good as the road distances in its knowledge base. These come from Practical Exercise 3 and are approximate driving distances between town centres, so a small error could change which route is cheapest.
*   **Static facts about a changing environment.** Road closures, weather and traffic change while the trailer drives. The agent only knows what we told it, so it must be given new facts (as in the landslide query) before it can replan.
*   **Fixed town names and coverage.** The agent can only match towns that already exist in the knowledge base. A typo or an unlisted town such as Mombasa raises an error, and many Kenyan towns and roads are not included.
*   **Unvalidated facts.** Early versions let a user add any fact, even a negative distance or a town with no region. We had to write the `audit` function to catch these.
*   **Assumed speed rules.** The speed limits for weather and hazards were chosen by the group, not taken from official regulations.

**Recommended improvements**
*   Retrieve distances and live road, traffic and weather data from a maps or weather API instead of typing them in, and refresh them while the trip is in progress.
*   Validate every new fact before it enters the knowledge base (positive distance, known region, valid weather and light values).
*   Make the path cost more realistic by using travel time, fuel use and road quality together, not distance alone.
*   Store the knowledge base in a file or database so facts persist between runs.
*   Move from dictionaries to logical rules (e.g. `If Weather(Rain) and Hazard(Escarpment) then SpeedLimit(40)`) with an inference engine, so the agent can explain and chain its conclusions.